In [1]:
# ============================================================
# CELL 1 (04_baselines): Baseline scores on the headline test
# What it does:
#   1) setup from disk: imports, src/metrics.py and src/splits.py, rulebook,
#      station_meta, model_base.parquet (rows with a valid target and valid
#      CHIRPS and ERA5-Land rain features), wide aligned rain for IDW
#   2) for every held-out station (7) and both fold schemes (blocked 30 km
#      and plain LOSO), scores on the HEADLINE rows (held-out station, 2011-2023):
#        raw CHIRPS, raw ERA5-Land, raw mean(CHIRPS, ERA5L)
#        QM local CHIRPS (oracle): quantile mapping fitted on the held-out
#          station's own 1983-2010 data, season by season
#        QM transfer CHIRPS / QM transfer mean: mapping fitted on pooled
#          training stations, 1983-2010, season by season
#        IDW gauges: inverse-distance (1/d^2) mean of the training stations'
#          same-day rain
#      Quantile mapping uses 103 quantile levels; above the largest fitted
#      satellite value, the last ratio is extended
#   3) saves outputs/baseline_scores.csv (all metrics, long format)
#   4) prints: summary by baseline and scheme (means over stations, plus
#      KGE for the 3 dry-side and 4 wet-side stations separately), then
#      per-station KGE and per-station percent bias
# ============================================================
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs"
sys.path.insert(0, str(ROOT / "src"))
import metrics, splits

with open(PROCESSED_DIR / "eda_decisions.json") as f:
    dec = json.load(f)
TRAIN_END = dec["temporal_split"]["train"][1]
THR = dec["extreme_thresholds_mm"]
meta = pd.read_csv(PROCESSED_DIR / "station_meta.csv", index_col="station")
stations = list(meta.index)
dist = splits.distance_matrix(meta)
blocked = splits.make_folds(stations, dist, "blocked", dec["spatial_split"]["block_km"])
loso = splits.make_folds(stations, dist, "loso")

rain_wide = pd.read_parquet(PROCESSED_DIR / "rain_final.parquet")[stations]
base = pd.read_parquet(PROCESSED_DIR / "model_base.parquet")
valid = (base.dropna(subset=["rain", "chirps_blend_c", "era_p_blend"])
             .sort_values(["station", "date"]).reset_index(drop=True))

SEASONS = ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"]
QGRID = np.concatenate([np.linspace(0, 0.99, 100), [0.995, 0.999, 1.0]])

chirps = valid["chirps_blend_c"].to_numpy()
era = valid["era_p_blend"].to_numpy()
avg = (chirps + era) / 2
obs = valid["rain"].to_numpy()
seas = valid["season"].to_numpy()
dates = valid["date"].to_numpy()

# --- quantile mapping ---
def fit_qm(sat, gauge):
    ok = np.isfinite(sat) & np.isfinite(gauge)
    s, g = sat[ok], gauge[ok]
    if len(s) < 200:
        return None
    sq = np.quantile(s, QGRID) + np.arange(len(QGRID)) * 1e-9      # strictly increasing
    gq = np.quantile(g, QGRID)
    return sq, gq

def apply_qm(x, fit):
    sq, gq = fit
    out = np.interp(x, sq, gq)
    hi = x > sq[-1]
    out[hi] = gq[-1] * x[hi] / sq[-1]
    return out

def qm_predict(var, train_mask, target_mask):
    sim = np.full(len(valid), np.nan)
    for s in SEASONS:
        tm = train_mask & (seas == s)
        fit = fit_qm(var[tm], obs[tm])
        t = target_mask & (seas == s)
        if fit is not None and t.any():
            sim[t] = apply_qm(var[t], fit)
    return sim

def idw_predict(fold, target_mask):
    tr, h = fold["train"], fold["held_out"]
    w = 1.0 / dist.loc[h, tr].to_numpy() ** 2
    R = rain_wide[tr].reindex(pd.DatetimeIndex(dates[target_mask])).to_numpy()
    ok = np.isfinite(R)
    num = np.where(ok, np.nan_to_num(R) * w, 0.0).sum(axis=1)
    den = (ok * w).sum(axis=1)
    sim = np.full(len(valid), np.nan)
    sim[target_mask] = np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)
    return sim

# --- scoring loop ---
results = []
def add(name, scheme, h, sim, mask):
    sc = metrics.score_all(dates[mask], obs[mask], sim[mask], THR[h])
    results.append({"baseline": name, "scheme": scheme, "station": h, **sc})

for scheme, folds in (("blocked", blocked), ("loso", loso)):
    for fo in folds:
        h = fo["held_out"]
        m = splits.fold_masks(valid, fo, TRAIN_END)
        hd = m["headline"]
        if scheme == "blocked":                       # scheme-independent baselines, scored once
            add("raw CHIRPS", "none", h, chirps, hd)
            add("raw ERA5-Land", "none", h, era, hd)
            add("raw mean(CHIRPS,ERA5L)", "none", h, avg, hd)
            add("QM local CHIRPS (oracle)", "none", h, qm_predict(chirps, m["spatial_only"], hd), hd)
        add("QM transfer CHIRPS", scheme, h, qm_predict(chirps, m["train"], hd), hd)
        add("QM transfer mean(CHIRPS,ERA5L)", scheme, h, qm_predict(avg, m["train"], hd), hd)
        add("IDW gauges (1/d^2)", scheme, h, idw_predict(fo, hd), hd)

res = pd.DataFrame(results)
res.to_csv(OUTPUT_DIR / "baseline_scores.csv", index=False)

# --- summaries ---
DRY = ["Kunchha", "Bandipur", "Damauli"]
short = {"raw CHIRPS": "rawCHIRPS", "raw ERA5-Land": "rawERA5L", "raw mean(CHIRPS,ERA5L)": "rawMean",
         "QM local CHIRPS (oracle)": "QMlocal*", "QM transfer CHIRPS": "QMt-CHIRPS",
         "QM transfer mean(CHIRPS,ERA5L)": "QMt-mean", "IDW gauges (1/d^2)": "IDW"}
res["label"] = [short[b] if s == "none" else f"{short[b]}/{s[:4]}"
                for b, s in zip(res["baseline"], res["scheme"])]
res["abs_pbias"] = res["pbias"].abs()
order = ["rawCHIRPS", "rawERA5L", "rawMean", "QMlocal*", "QMt-CHIRPS/bloc", "QMt-CHIRPS/loso",
         "QMt-mean/bloc", "QMt-mean/loso", "IDW/bloc", "IDW/loso"]

g = res.groupby("label")
summ = g[["kge", "r", "abs_pbias", "wet_far", "q95_ratio", "rx1_ratio", "p95_pod"]].mean()
summ["kge_dry3"] = res[res["station"].isin(DRY)].groupby("label")["kge"].mean()
summ["kge_wet4"] = res[~res["station"].isin(DRY)].groupby("label")["kge"].mean()
summ = summ.loc[order, ["kge", "kge_dry3", "kge_wet4", "r", "abs_pbias", "wet_far",
                        "q95_ratio", "rx1_ratio", "p95_pod"]].round(2)
print("Headline test (held-out station, 2011-2023). Means over the 7 stations;")
print("dry3 = Kunchha, Bandipur, Damauli; wet4 = the other four. QMlocal* uses the held-out station's own gauge data.")
print(summ.to_string())

for metric, nd, title in (("kge", 2, "KGE"), ("pbias", 0, "percent bias of total rainfall")):
    pv = res.pivot_table(index="station", columns="label", values=metric).loc[stations, order].round(nd)
    print(f"\nPer-station {title}:")
    print(pv.to_string())
print("\nSaved: outputs/baseline_scores.csv")

Headline test (held-out station, 2011-2023). Means over the 7 stations;
dry3 = Kunchha, Bandipur, Damauli; wet4 = the other four. QMlocal* uses the held-out station's own gauge data.
                  kge  kge_dry3  kge_wet4     r  abs_pbias  wet_far  q95_ratio  rx1_ratio  p95_pod
label                                                                                             
rawCHIRPS        0.41      0.37      0.43  0.49       6.35     0.41       0.63       0.69     0.10
rawERA5L         0.31      0.18      0.41  0.50      37.05     0.52       0.71       0.79     0.15
rawMean          0.41      0.37      0.45  0.53      19.92     0.50       0.61       0.67     0.10
QMlocal*         0.43      0.42      0.43  0.44       6.81     0.31       1.11       1.14     0.23
QMt-CHIRPS/bloc  0.30      0.39      0.23  0.44      22.20     0.31       1.22       1.20     0.25
QMt-CHIRPS/loso  0.40      0.40      0.40  0.44      12.06     0.31       1.04       1.07     0.21
QMt-mean/bloc    0.42    